In [15]:
#Iterative Deepening DFS Task 3

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}

class GoalBasedAgent:
    def __init__(self, goal) -> None:
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return 'Goal Reached'
        else:
            return 'Searching'
        
    def dfs_limited(self, graph, start, goal, depth, visited, path, parent):
        visited.append(start)
        path.append(start)
        parent[start] = path[-2] if len(path) > 1 else None
        
        print(f'Visited: {start} at depth {depth}')
        
        if start == goal:
            return True
        
        if depth > 0:
            for child in graph[start]:
                if child not in visited:
                    if self.dfs_limited(graph, child, goal, depth - 1, visited, path, parent):
                        return True
        
        return False
    
    def iterative_deepening(self, graph, start, goal):
        depth = 0
        max_depth = len(graph)
        
        while depth <= max_depth:
            print(f'\n--- Searching at depth limit: {depth} ---')
            visited = []
            path = []
            parent = {}
            
            if self.dfs_limited(graph, start, goal, depth, visited, path, parent):
                route = []
                current = goal
                while current is not None:
                    route.append(current)
                    current = parent.get(current)
                route.reverse()
                return f'Goal Reached\nPath: {"->".join(route)}'
            
            depth += 1
        
        return 'Goal Not Found'
            
    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == 'Goal Reached':
            return f'Goal {self.goal} found!'
        else:
            return self.iterative_deepening(graph, percept, self.goal)
        
class Environment:
    def __init__(self, graph) -> None:
        self.graph = graph

    def get_percept(self, node):
        return node
    

def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)
    print(action)

start_node = 'A'
goal_node = 'I'

agent = GoalBasedAgent(goal_node)
env = Environment(graph)

run_agent(agent, env, start_node)


--- Searching at depth limit: 0 ---
Visited: A at depth 0

--- Searching at depth limit: 1 ---
Visited: A at depth 1
Visited: B at depth 0
Visited: C at depth 0

--- Searching at depth limit: 2 ---
Visited: A at depth 2
Visited: B at depth 1
Visited: D at depth 0
Visited: E at depth 0
Visited: C at depth 1
Visited: F at depth 0
Visited: G at depth 0

--- Searching at depth limit: 3 ---
Visited: A at depth 3
Visited: B at depth 2
Visited: D at depth 1
Visited: H at depth 0
Visited: E at depth 1
Visited: C at depth 2
Visited: F at depth 1
Visited: I at depth 0
Goal Reached
Path: A->B->D->H->E->C->F->I


In [ ]:
#TSP Task 2

cost_matrix = {
    1: {1: 0, 2: 10, 3: 15, 4: 20},
    2: {1: 10, 2: 0, 3: 35, 4: 25},
    3: {1: 15, 2: 35, 3: 0, 4: 30},
    4: {1: 20, 2: 25, 3: 30, 4: 0}
}

class GoalBasedAgent:
    def __init__(self, start_city) -> None:
        self.start_city = start_city
        self.best_path = None
        self.best_cost = float('inf')

    def formulate_goal(self, percept):
        if percept == 'find_tsp':
            return 'Searching'
        else:
            return 'Goal Reached'
        
    def calculate_path_cost(self, path, cost_matrix):
        total_cost = 0
        for i in range(len(path) - 1):
            total_cost += cost_matrix[path[i]][path[i + 1]]
        return total_cost
    
    def generate_permutations(self, cities):
        if len(cities) == 0:
            return [[]]
        if len(cities) == 1:
            return [[cities[0]]]
        
        perms = []
        for i in range(len(cities)):
            current = cities[i]
            remaining = cities[:i] + cities[i+1:]
            for perm in self.generate_permutations(remaining):
                perms.append([current] + perm)
        return perms
    
    def tsp(self, cost_matrix, start_city):
        cities = [city for city in cost_matrix.keys() if city != start_city]
        permutations = self.generate_permutations(cities)
        
        for perm in permutations:
            path = [start_city] + perm + [start_city]
            cost = self.calculate_path_cost(path, cost_matrix)
            print(f'Path: {path} Cost: {cost}')
            
            if cost < self.best_cost:
                self.best_cost = cost
                self.best_path = path
        
        return f'Best Path: {self.best_path}\nMinimum Cost: {self.best_cost}'
            
    def act(self, percept, cost_matrix):
        goal_status = self.formulate_goal(percept)
        if goal_status == 'Goal Reached':
            return f'Goal {self.start_city} found!'
        else:
            return self.tsp(cost_matrix, self.start_city)
        
class Environment:
    def __init__(self, cost_matrix) -> None:
        self.cost_matrix = cost_matrix

    def get_percept(self):
        return 'find_tsp'
    

def run_agent(agent, environment):
    percept = environment.get_percept()
    action = agent.act(percept, environment.cost_matrix)
    print(action)

start_city = 1

agent = GoalBasedAgent(start_city)
env = Environment(cost_matrix)

run_agent(agent, env)

Path: [1, 2, 3, 4, 1] Cost: 95
Path: [1, 2, 4, 3, 1] Cost: 80
Path: [1, 3, 2, 4, 1] Cost: 95
Path: [1, 3, 4, 2, 1] Cost: 80
Path: [1, 4, 2, 3, 1] Cost: 95
Path: [1, 4, 3, 2, 1] Cost: 95
Best Path: [1, 2, 4, 3, 1]
Minimum Cost: 80


In [11]:
#UCS Goal Based Agent Task 1
import heapq

graph = {
    'A': [('B', 3), ('C', 4)],
    'B': [('D', 1), ('E', 5)],
    'C': [('F', 2), ('G', 3)],
    'D': [('H', 1)],
    'E': [],
    'F': [('I', 4)],
    'G': [],
    'H': [],
    'I': []
}

class GoalBasedAgent:
    def __init__(self, goal) -> None:
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return 'Goal Reached'
        else:
            return 'Searching'
        
    def ucs(self, graph, start, goal):
        visited = set()
        priority_queue = []
        
        heapq.heappush(priority_queue, (0, start))
        
        while priority_queue:
            cost, node = heapq.heappop(priority_queue)
            print(f'Visited: {node} with cost {cost}')
            
            if node in visited:
                continue
            
            visited.add(node)
            
            if node == goal:
                return f'Goal Reached with cost {cost}'
            
            for child, edge_cost in graph[node]:
                if child not in visited:
                    total_cost = cost + edge_cost
                    heapq.heappush(priority_queue, (total_cost, child))
        
        return 'Goal Not Found'
            
    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == 'Goal Reached':
            return f'Goal {self.goal} found!'
        else:
            return self.ucs(graph, percept, self.goal)
        
class Environment:
    def __init__(self, graph) -> None:
        self.graph = graph

    def get_percept(self, node):
        return node
    

def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)
    print(action)

start_node = 'A'
goal_node = 'I'

agent = GoalBasedAgent(goal_node)
env = Environment(graph)

run_agent(agent, env, start_node)

Visited: A with cost 0
Visited: B with cost 3
Visited: C with cost 4
Visited: D with cost 4
Visited: H with cost 5
Visited: F with cost 6
Visited: G with cost 7
Visited: E with cost 8
Visited: I with cost 10
Goal Reached with cost 10


In [13]:
#DLS Goal Based Agent Task 1
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}

class GoalBasedAgent:
    def __init__(self, goal, depth_limit) -> None:
        self.goal = goal
        self.depth_limit = depth_limit

    def formulate_goal(self, percept):
        if percept == self.goal:
            return 'Goal Reached'
        else:
            return 'Searching'
        
    def dls(self, graph, start, goal, depth):
        visited = []
        stack = []
        
        visited.append(start)
        stack.append((start, 0))
        
        while stack:
            node, current_depth = stack.pop()
            print(f'Visited: {node} at depth {current_depth}')
            
            if node == goal:
                return 'Goal Reached'
            
            if current_depth < depth:
                for child in reversed(graph[node]):
                    if child not in visited:
                        visited.append(child)
                        stack.append((child, current_depth + 1))
        
        return 'Goal Not Found'
            
    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == 'Goal Reached':
            return f'Goal {self.goal} found!'
        else:
            return self.dls(graph, percept, self.goal, self.depth_limit)
        
class Environment:
    def __init__(self, graph) -> None:
        self.graph = graph

    def get_percept(self, node):
        return node
    

def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)
    print(action)

start_node = 'A'
goal_node = 'F'
depth_limit = 1

agent = GoalBasedAgent(goal_node, depth_limit)
env = Environment(graph)

run_agent(agent, env, start_node)

Visited: A at depth 0
Visited: B at depth 1
Visited: C at depth 1
Goal Not Found


In [4]:
#DFS
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}

class GoalBasedAgent:
    def __init__(self, goal) -> None:
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return 'Goal Reached'
        else:
            return 'Searching'
        
    def dfs(self, graph, start, goal):
        visited = []
        stack = []

        visited.append(start)
        stack.append(start)

        while stack:
            node = stack.pop()
            print('Visited:', node)

            if node == goal:
                return 'Goal Reached'
            
            for child in reversed(graph[node]):
                if child not in visited:
                    visited.append(child)
                    stack.append(child)
                    
        return 'Goal Not Found'
            
    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == 'Goal Reached':
            return f'Goal {self.goal} found!'
        else:
            return self.dfs(graph, percept, self.goal)
        
class Environment:
    def __init__(self, graph) -> None:
        self.graph = graph

    def get_percept(self, node):
        return node
    

def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)
    print(action)

start_node = 'A'
goal_node = 'I'

agent = GoalBasedAgent(goal_node)
env = Environment(graph)

run_agent(agent, env, start_node)

Visited: A
Visited: B
Visited: D
Visited: H
Visited: E
Visited: C
Visited: F
Visited: I
Goal Reached


In [ ]:
#BFS Maze - Agent-Based Model

maze = [
    [1, 1, 0],
    [1, 1, 1],
    [0, 1, 1]
]

directions = [(0, 1), (1, 0)]

def create_graph(maze):
    graph = {}
    rows = len(maze)
    cols = len(maze[0])

    for i in range(rows):
        for j in range(cols):
            if maze[i][j] == 1:
                neighbours = []
                for dx, dy in directions:
                    nx, ny = i + dx, j + dy
                    if 0<= nx < rows and 0 <= ny < cols and maze[nx][ny] == 1:
                        neighbours.append((nx, ny))
                graph[(i, j)] = neighbours

    return graph

class GoalBasedAgent:
    def __init__(self, goal) -> None:
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return 'Goal Reached'
        else:
            return 'Searching'
        
    def bfs(self, graph, start, goal):
        visited = []
        queue = []

        visited.append(start)
        queue.append(start)

        while queue:
            node = queue.pop(0)
            print('Visited:', node)

            if node == goal:
                return 'Goal Reached'
            
            for child in graph[node]:
                if child not in visited:
                    visited.append(child)
                    queue.append(child)
        
        return 'Goal Not Found'
            
    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == 'Goal Reached':
            return f'Goal {self.goal} found!'
        else:
            return self.bfs(graph, percept, self.goal)
        
class Environment:
    def __init__(self, graph) -> None:
        self.graph = graph

    def get_percept(self, node):
        return node
    

def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)
    print(action)

graph = create_graph(maze)

start_node = (0, 0)
goal_node = (2, 2)

agent = GoalBasedAgent(goal_node)
env = Environment(graph)

run_agent(agent, env, start_node)
        


Visited: (0, 0)
Visited: (0, 1)
Visited: (1, 0)
Visited: (1, 1)
Visited: (1, 2)
Visited: (2, 1)
Visited: (2, 2)


'Goal Reached'

In [ ]:
#BFS Goal Based Agent

tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}
class GoalBasedAgent:
    def __init__(self, goal):
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal Reached"
        return "Searching"
    
    def bfs_search(self, tree, start, goal):
        visited = []
        queue = []

        visited.append(start)
        queue.append(start)

        while queue:
            node = queue.pop(0)
            print('Visited:', node)

            if node == goal:
                print("Goal Found")
                return
            
            for neighbour in tree[node]:
                if neighbour not in visited:
                    visited.append(neighbour)
                    queue.append(neighbour)

    def act(self, percept, tree):
        action = self.formulate_goal(percept)
        if action == 'Goal Reached':
            return f'{self.goal} is found'
        else:
            self.bfs_search(tree, percept, self.goal)


class Environment:
    def __init__(self, graph):
        self.graph = graph

    def get_percept(self, node):
        return node
    

def run_agent(agent, environment, start):
    percept = environment.get_percept(start)
    action = agent.act(percept, environment.graph)
    print(action)

start_node = 'A'
goal_node = 'H'

g = GoalBasedAgent(goal_node)
e = Environment(tree)

run_agent(g, e, start_node)




Visited: A
Visited: B
Visited: C
Visited: D
Visited: E
Visited: F
Visited: G
Visited: H
Goal Found
